## CNN
The MNIST dataset is a large collection of handwritten digits. It’s widely regarded as a benchmark dataset for evaluating the performance of various image classification algorithms. The dataset contains 70,000 grayscale images of handwritten digits (0 to 9), with each image having a size of 28x28 pixels. The pixel values range from 0 to 255, where 0 represents black, 255 represents white, and the values in between represent various shades of gray.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
%matplotlib inline

In [ ]:
import tensorflow as tf
from tensorflow.keras.datasets import mnist

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [ ]:
sample_set = x_train[0]

In [ ]:
def calculate_target(img_shape, kernel_shape):
    row_size = img_shape[0] - kernel_shape[0] + 1
    col_size = img_shape[1] - kernel_shape[1] + 1
    target = np.zeros((row_size , col_size))
    
    return row_size, col_size, target

def convolve(img: np.array, kernel: np.array) -> np.array:
    col_size, row_size, CovImage = calculate_target(
        img.shape,
        kernel.shape
    )

    for i in range(col_size):
        for j in range(row_size):
            CovImage[i,j] = (img[i:i+kernel.shape[0],j:j+kernel.shape[1]] * kernel).sum()

    return CovImage

In [ ]:
kersl = np.array([[0, 0, 0],
                   [0, 0, 1],
                   [0, 0, 0]])


In [ ]:
## compare show
for _ in x_train[:10]:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(_, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    convolved_image = convolve(_, kersl)
    
    plt.subplot(1, 2, 2)
    plt.imshow(convolved_image, cmap='gray')
    plt.title('Convolved Image')
    plt.axis('off')

    plt.show()

In [ ]:
kerst = np.array([[0, 0, 0],
                   [0, 0, 0],
                   [0, 1, 0]])
## compare show
for _ in x_train[:10]:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(_, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    convolved_image = convolve(_, kerst)
    
    plt.subplot(1, 2, 2)
    plt.imshow(convolved_image, cmap='gray')
    plt.title('Convolved Image')
    plt.axis('off')

    plt.show()

In [ ]:
kerlap = np.array([[0, -2, 0],
                   [-2, 8, -2],
                   [0, -2, 0]])/2
for _ in x_train[:10]:
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(_, cmap='gray')
    plt.title('Original Image')
    plt.axis('off')

    convolved_image = convolve(_, kerlap)
    
    plt.subplot(1, 2, 2)
    plt.imshow(convolved_image, cmap='gray')
    plt.title('Convolved Image')
    plt.axis('off')

    plt.show()

### Tensorflow version


Using Tensorflow to fit a CNN which performs 10 class classification, using the images in x_train to predict y_train and evaluate performace on the test data. 

In [25]:
## Define the ResNet
def resblock(x, filters, kernel_size):
    shortcut = x
    x = tf.keras.layers.Conv2D(filters, (kernel_size, kernel_size), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = tf.keras.layers.Conv2D(filters, (kernel_size, kernel_size), padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Add()([shortcut, x])
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)

    return x


In [48]:
## build a model with 2 hidden layers with cov2d
inputs = tf.keras.Input(shape = (28, 28))
x = tf.keras.layers.Reshape((28, 28, 1))(inputs)
x = tf.keras.layers.Conv2D(32, (3, 3), padding='same')(x)
x = tf.keras.layers.Activation('relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D((2, 2))(x)
x = tf.keras.layers.Conv2D(64, (3, 3), padding='same')(x)
x = tf.keras.layers.Activation('relu')(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.MaxPooling2D((2, 2))(x)
x = tf.keras.layers.Flatten()(x)
x = tf.keras.layers.Dense(64)(x)
outputs = tf.keras.layers.Dense(10, activation='softmax')(x)

model = tf.keras.Model(inputs = inputs, outputs = outputs)

In [53]:
fit_sample = tf.keras.Sequential([
    tf.keras.Input(shape = (28, 28)),
    tf.keras.layers.Reshape((28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), padding='same'),
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64),
    tf.keras.layers.Dense(10, activation='softmax')
])
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
fit_sample.compile(optimizer=opt, 
              loss='sparse_categorical_crossentropy',
              metrics = ['accuracy'])
fit_sample.fit(x_train, y_train, batch_size=32, epochs=10, validation_data=(x_test, y_test))

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9431 - loss: 0.4192 - val_accuracy: 0.9736 - val_loss: 0.1227
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9774 - loss: 0.0945 - val_accuracy: 0.9763 - val_loss: 0.0875
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9822 - loss: 0.0595 - val_accuracy: 0.9756 - val_loss: 0.0961
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9845 - loss: 0.0502 - val_accuracy: 0.9776 - val_loss: 0.0874
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9865 - loss: 0.0426 - val_accuracy: 0.9761 - val_loss: 0.0870
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9885 - loss: 0.0358 - val_accuracy: 0.9777 - val_loss: 0.0871
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9898 - loss: 0.0303 - val_accuracy: 0.9766 - val_loss: 0.1043
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 7ms/step - accuracy: 0.9908 - loss: 0

In [49]:
indices = np.random.permutation(len(x_train))
x_shuffle = x_train[indices]
y_shuffle = y_train[indices]
x_sample = x_shuffle[:10000]
y_sample = x_shuffle[:10000]

In [51]:
opt = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=opt, 
              loss='sparse_categorical_crossentropy',
              metrics = ['accuracy'])
model.fit(x_train, y_train, batch_size=32, epochs=10, validation_data=(x_test, y_test))

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.9792 - loss: 0.0894 - val_accuracy: 0.9820 - val_loss: 0.0773
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9866 - loss: 0.0470 - val_accuracy: 0.9818 - val_loss: 0.0641
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.9897 - loss: 0.0357 - val_accuracy: 0.9842 - val_loss: 0.0558
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - accuracy: 0.9910 - loss: 0.0289 - val_accuracy: 0.9894 - val_loss: 0.0373
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 20s 11ms/step - accuracy: 0.9922 - loss: 0.0256 - val_accuracy: 0.9871 - val_loss: 0.0484
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9939 - loss: 0.0199 - val_accuracy: 0.9861 - val_loss: 0.0509
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9941 - loss: 0.0193 - val_accuracy: 0.9880 - val_loss: 0.0438
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 10ms/step - accuracy: 0.9951 - 

### Using torch

In [55]:
import torch.functional as F
import torch.nn as nn